In [1]:
import os

os.makedirs("../csv", exist_ok=True)
os.makedirs("../svg", exist_ok=True)

In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import scienceplots

In [3]:
plt.rcParams.update({
    'font.family': 'TeX Gyre Termes',
    'font.size': 11,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'figure.dpi': 300,
    'savefig.dpi': 300,
})

In [4]:
TEXTWIDTH_PT = 426.79137
TEXTHEIGHT_PT = 702.78308
PT_PER_INCH = 72.27

FIG_WIDTH = TEXTWIDTH_PT / PT_PER_INCH
FIG_HEIGHT_1 = TEXTHEIGHT_PT / PT_PER_INCH
FIG_HEIGHT_2 = FIG_HEIGHT_1 / 2
FIG_HEIGHT_3 = FIG_HEIGHT_1 / 3
FIG_HEIGHT_4 = FIG_HEIGHT_1 / 4
FIG_HEIGHT_5 = FIG_HEIGHT_1 / 5
FIG_HEIGHT_6 = FIG_HEIGHT_1 / 6
FIG_HEIGHT_7 = FIG_HEIGHT_1 / 7
FIG_HEIGHT_8 = FIG_HEIGHT_1 / 8

In [5]:
backendy_pl = {
    'scalar': 'skalarny',
    'sse2': 'SSE2',
    'avx2': 'AVX2',
    'avx512': 'AVX-512',
    'neon': 'NEON',
}

In [6]:
wersje_pl = {
    'Speck32_64':   '32/64',
    'Speck48_72':   '48/72',
    'Speck48_96':   '48/96',
    'Speck64_96':   '64/96',
    'Speck64_128':  '64/128',
    'Speck96_96':   '96/96',
    'Speck96_144':  '96/144',
    'Speck128_128': '128/128',
    'Speck128_192': '128/192',
    'Speck128_256': '128/256',
}

In [7]:
backend_order = ['scalar', 'sse2', 'avx2', 'avx512']
version_order = ['32_64', '48_72', '48_96', '64_96', '64_128', '96_96', '96_144', '128_128', '128_192', '128_256']

In [8]:
system_x86 = pd.read_csv('../data/system_x86.csv')
system_aarch64 = pd.read_csv('../data/system_aarch64.csv')

In [9]:
system_x86['time_per_key_ns'] = system_x86['duration_ns'] / system_x86['throughput_num']
system_x86['keys_per_sec'] = 1e9 / system_x86['time_per_key_ns']

print(system_x86)

      bits_measured benchmark backend architecture         function  \
0                12    system  Avx512       x86_64  EncryptInflight   
1                13    system  Avx512       x86_64  EncryptInflight   
2                14    system  Avx512       x86_64  EncryptInflight   
3                15    system  Avx512       x86_64  EncryptInflight   
4                16    system  Avx512       x86_64  EncryptInflight   
...             ...       ...     ...          ...              ...   
1115             22    system  Scalar       x86_64  EncryptInflight   
1116             23    system  Scalar       x86_64  EncryptInflight   
1117             24    system  Scalar       x86_64  EncryptInflight   
1118             28    system  Scalar       x86_64  EncryptInflight   
1119             32    system  Scalar       x86_64  EncryptInflight   

           version  suffix  throughput_num unit  duration_ns  time_per_key_ns  \
0       Speck32_64       1            4096   ns       662920      

In [10]:
system_aarch64

,bits_measured,benchmark,backend,architecture,function,version,suffix,throughput_num,unit,duration_ns
0,11,system,Neon,aarch64,EncryptInflight,Speck32_64,1,2048,ns,633000
1,12,system,Neon,aarch64,EncryptInflight,Speck32_64,1,4096,ns,246375
2,13,system,Neon,aarch64,EncryptInflight,Speck32_64,1,8192,ns,269791
3,14,system,Neon,aarch64,EncryptInflight,Speck32_64,1,16384,ns,298458
4,15,system,Neon,aarch64,EncryptInflight,Speck32_64,1,32768,ns,335417
...,...,...,...,...,...,...,...,...,...,...
555,21,system,Scalar,aarch64,EncryptInflight,Speck128_256,2,2097152,ns,10904750
556,22,system,Scalar,aarch64,EncryptInflight,Speck128_256,2,4194304,ns,20637250
557,23,system,Scalar,aarch64,EncryptInflight,Speck128_256,2,8388608,ns,40437083
558,27,system,Scalar,aarch64,EncryptInflight,Speck128_256,2,134217728,ns,616849208


In [11]:
import pandas as pd
import numpy as np
from scipy import stats

system_x86 = pd.read_csv('../data/system_x86.csv')

GROUP_COLS = ['backend', 'function', 'version', 'suffix']

def key_bits_from_version(v):
    return int(str(v).split('_')[1])

def fit_group(g):
    g = g.sort_values('throughput_num')
    n = g['throughput_num'].to_numpy(dtype=float)
    t = g['duration_ns'].to_numpy(dtype=float)

    if len(g) < 3:
        return None

    kb = key_bits_from_version(g['version'].iloc[0])
    N_full = 2.0 ** kb

    slope, intercept, lo, hi = stats.theilslopes(t, n)
    t_key = slope
    thr = 1e9 / t_key if t_key > 0 else np.nan

    tau, p_tau = stats.kendalltau(n, t)

    pred_total_ns = intercept + slope * N_full
    pred_total_s = pred_total_ns * 1e-9
    span = n.max() / n.min()

    return {
        'key_bits': kb,
        'overhead_ns': intercept,
        'ns_per_key': t_key,
        'ns_per_key_lo': 1e9 / hi if hi > 0 else np.nan,
        'ns_per_key_hi': 1e9 / lo if lo > 0 else np.nan,
        'throughput_keys_per_s': thr,
        'kendall_tau': tau,
        'p_value_kendall': p_tau,
        'full_keyspace': N_full,
        'pred_total_seconds': pred_total_s,
        'pred_total_years': pred_total_s / (3600 * 24 * 365.25),
        'reliable': (t_key > 0) and (span >= 8) and (p_tau < 0.05),
    }

rows = []
for keys, g in system_x86.groupby(GROUP_COLS):
    res = fit_group(g)
    if res is not None:
        rows.append(dict(zip(GROUP_COLS, keys)) | res)

predictions = pd.DataFrame(rows).sort_values(GROUP_COLS).reset_index(drop=True)
predictions

,backend,function,version,suffix,key_bits,overhead_ns,ns_per_key,ns_per_key_lo,ns_per_key_hi,throughput_keys_per_s,kendall_tau,p_value_kendall,full_keyspace,pred_total_seconds,pred_total_years,reliable
0,Avx2,EncryptInflight,Speck128_128,1,128,137628.571429,1.347395,5.435785e+08,9.005371e+08,7.421730e+08,0.823516,0.000068,3.402824e+38,4.584946e+29,1.452882e+22,True
1,Avx2,EncryptInflight,Speck128_128,2,128,170153.000038,1.094840,8.708714e+08,9.175422e+08,9.133751e+08,0.937893,0.000006,3.402824e+38,3.725549e+29,1.180555e+22,True
2,Avx2,EncryptInflight,Speck128_192,1,192,143078.142857,1.129421,5.692918e+08,8.981668e+08,8.854097e+08,0.777765,0.000168,6.277102e+57,7.089489e+48,2.246523e+41,True
3,Avx2,EncryptInflight,Speck128_192,2,192,-288536.945113,1.158503,8.585931e+08,9.046162e+08,8.631829e+08,0.960769,0.000003,6.277102e+57,7.272041e+48,2.304371e+41,True
4,Avx2,EncryptInflight,Speck128_256,1,256,130360.488851,1.119126,8.656569e+08,8.962663e+08,8.935547e+08,0.846392,0.000042,1.157921e+77,1.295859e+68,4.106329e+60,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,Sse2,EncryptInflight,Speck64_96,2,96,168335.500000,0.843402,1.132978e+09,1.189912e+09,1.185675e+09,0.960769,0.000003,7.922816e+28,6.682116e+19,2.117435e+12,True
76,Sse2,EncryptInflight,Speck96_144,1,144,132779.044454,2.857039,3.492459e+08,3.539951e+08,3.500127e+08,0.800641,0.000108,2.230075e+43,6.371410e+34,2.018978e+27,True
77,Sse2,EncryptInflight,Speck96_144,2,144,931151.071429,2.694653,3.414177e+08,3.714163e+08,3.711053e+08,0.960769,0.000003,2.230075e+43,6.009276e+34,1.904225e+27,True
78,Sse2,EncryptInflight,Speck96_96,1,96,138558.535378,2.758620,3.501254e+08,4.055245e+08,3.625001e+08,0.846392,0.000042,7.922816e+28,2.185604e+20,6.925761e+12,True


In [12]:
import os
import numpy as np

INDEX_COLS = ["backend", "version"]
SUFFIXES   = sorted(predictions["suffix"].unique())

def fmt_pow2(x):
    if pd.isna(x) or x <= 0:
        return "--"
    return rf"\(2^{{{np.log2(x):.2f}}}\)"

def fmt_sci(x):
    if pd.isna(x):
        return "--"
    m, e = f"{x:.3e}".split("e")
    return rf"\({float(m):.2f}\times10^{{{int(e)}}}\)"

wide = predictions.set_index(INDEX_COLS + ["suffix"])

cols = {}
for suf in SUFFIXES:
    sub = wide.xs(suf, level="suffix")
    cols[f"{suf}_throughput"] = sub["throughput_keys_per_s"].map(fmt_pow2)
    cols[f"{suf}_years"]      = sub["pred_total_years"].map(fmt_sci)

latex_df = pd.DataFrame(cols).reset_index()

# --- klucze sortujące z SUROWYCH wartości (przed tłumaczeniem) ---
latex_df["_bk"] = pd.Categorical(
    latex_df["backend"].astype(str).str.lower(),
    categories=backend_order, ordered=True)
latex_df["_ver"] = pd.Categorical(
    latex_df["version"].astype(str).str.replace("Speck", "", regex=False),
    categories=version_order, ordered=True)

latex_df = latex_df.sort_values(["_bk", "_ver"]).drop(columns=["_bk", "_ver"])

# --- dopiero teraz tłumaczenia na tekst ---
latex_df["backend"] = latex_df["backend"].astype(str).str.lower().map(backendy_pl).fillna(latex_df["backend"])
latex_df["version"] = latex_df["version"].map(wersje_pl).fillna(latex_df["version"])

latex_df = latex_df.reset_index(drop=True)

os.makedirs("../csv", exist_ok=True)
latex_df.to_csv("../csv/prediction_results.csv", index=False)
latex_df

,backend,version,1_throughput,1_years,2_throughput,2_years
0,skalarny,32/64,\(2^{29.37}\),\(8.44\times10^{2}\),\(2^{29.36}\),\(8.48\times10^{2}\)
1,skalarny,48/72,\(2^{28.37}\),\(4.33\times10^{5}\),\(2^{28.35}\),\(4.38\times10^{5}\)
2,skalarny,48/96,\(2^{28.13}\),\(8.54\times10^{12}\),\(2^{28.30}\),\(7.60\times10^{12}\)
3,skalarny,64/96,\(2^{28.85}\),\(5.20\times10^{12}\),\(2^{28.81}\),\(5.34\times10^{12}\)
4,skalarny,64/128,\(2^{28.89}\),\(2.17\times10^{22}\),\(2^{28.87}\),\(2.20\times10^{22}\)
5,skalarny,96/96,\(2^{27.78}\),\(1.09\times10^{13}\),\(2^{27.77}\),\(1.09\times10^{13}\)
6,skalarny,96/144,\(2^{27.55}\),\(3.59\times10^{27}\),\(2^{27.68}\),\(3.29\times10^{27}\)
7,skalarny,128/128,\(2^{28.76}\),\(2.38\times10^{22}\),\(2^{28.69}\),\(2.49\times10^{22}\)
8,skalarny,128/192,\(2^{28.40}\),\(5.60\times10^{41}\),\(2^{28.47}\),\(5.35\times10^{41}\)
9,skalarny,128/256,\(2^{28.57}\),\(9.19\times10^{60}\),\(2^{28.54}\),\(9.43\times10^{60}\)


In [13]:
import numpy as np

# różnica log2 throughputu: dodatnia => suffix 2 szybszy
piv = predictions.pivot_table(
    index=["backend", "version"],
    columns="suffix",
    values="throughput_keys_per_s",
    observed=True,
)

cmp = piv.reset_index()
cmp.columns = ["backend", "version", "thr_s1", "thr_s2"]

cmp["delta_log2"]        = np.log2(cmp["thr_s2"] / cmp["thr_s1"])   # +1.0 = 2x szybciej
cmp["speedup_s2_over_s1"] = cmp["thr_s2"] / cmp["thr_s1"]           # krotność
cmp["winner"] = np.where(cmp["delta_log2"] > 0, "suffix 2",
                np.where(cmp["delta_log2"] < 0, "suffix 1", "remis"))

suffix_analysis = cmp.sort_values("delta_log2", ascending=False).reset_index(drop=True)
suffix_analysis

,backend,version,thr_s1,thr_s2,delta_log2,speedup_s2_over_s1,winner
0,Avx512,Speck32_64,1.609004e+09,6.934569e+09,2.107638,4.309852,suffix 2
1,Avx2,Speck32_64,1.807741e+09,3.999731e+09,1.145715,2.212558,suffix 2
2,Avx512,Speck64_128,1.629571e+09,2.873268e+09,0.818200,1.763205,suffix 2
3,Avx512,Speck64_96,1.749303e+09,2.794973e+09,0.676054,1.597763,suffix 2
4,Avx2,Speck48_72,1.129377e+09,1.407474e+09,0.317581,1.246240,suffix 2
5,Avx2,Speck96_144,5.677521e+08,7.070456e+08,0.316542,1.245342,suffix 2
6,Avx2,Speck128_128,7.421730e+08,9.133751e+08,0.299452,1.230677,suffix 2
7,Sse2,Speck32_64,2.123272e+09,2.499299e+09,0.235235,1.177098,suffix 2
8,Scalar,Speck48_96,2.939582e+08,3.302693e+08,0.168032,1.123525,suffix 2
9,Scalar,Speck96_144,1.968781e+08,2.145828e+08,0.124232,1.089927,suffix 2


In [15]:
wins = suffix_analysis["winner"].value_counts()

summary = pd.DataFrame({
    "metryka": [
        "wierszy ogółem", "wygrane suffix 2", "wygrane suffix 1", "remisy",
        "śr. przewaga suffix 2 [×]", "mediana przewagi [×]",
        "maks. przewaga suffix 2 [×]", "maks. przewaga suffix 1 [×]",
    ],
    "wartość": [
        len(suffix_analysis),
        int(wins.get("suffix 2", 0)),
        int(wins.get("suffix 1", 0)),
        int(wins.get("remis", 0)),
        2.0 ** suffix_analysis["delta_log2"].mean(),
        2.0 ** suffix_analysis["delta_log2"].median(),
        suffix_analysis["speedup_s2_over_s1"].max(),
        1.0 / suffix_analysis["speedup_s2_over_s1"].min(),
    ],
})

per_backend = (suffix_analysis
    .groupby("backend", observed=True)["delta_log2"]
    .agg(["mean", "median", "min", "max"])
    .assign(speedup_mean=lambda d: 2.0 ** d["mean"]))

per_backend

,mean,median,min,max,speedup_mean
backend,,,,,
Avx2,0.213472,0.062588,-0.078202,1.145715,1.159475
Avx512,0.388788,0.060174,-0.011275,2.107638,1.309293
Scalar,0.016019,-0.013700,-0.065852,0.168032,1.011165
Sse2,0.018667,0.012334,-0.210142,0.235235,1.013023
